# Düzenlileştirme

**Titanic** yolcularının hayatta kalma şansını etkileyen faktörlere dair anlayışımızı geliştirelim
- Yorumlaması kolay olan lojistik sınıflandırıcıları kullanacağız
- Bunu daha önce "Karar Bilimi - Lojistik Regresyon" dersinde statsmodels ile yapmıştık
- Hangi özelliklerin alakasız olduğunu / genelleştirilemediğini tespit etmek için `p-değerleri` ve istatistiksel varsayımlar kullanıyorduk
- Bu sefer, eksik/aşırı öğrenme kriterlerine dayalı olarak alakalı/alakasız özellikleri tespit etmek için `düzenlileştirme` kullanacağız
- **Amacımız `L1` ve `L2` cezalarını karşılaştırmak**

## 1. Veriyi sizin için yüklüyor ve ön işleme tabi tutuyoruz

In [7]:
import pandas as pd
import numpy as np

In [8]:
data = pd.read_csv("https://d32aokrjazspmn.cloudfront.net/materials/ML_titanic_dataset_encoded.csv")

# the dataset is already one-hot-encoded
data.head()

,survived,pclass,age,sibsp,parch,fare,sex_female,class_First,class_Third,who_child,embark_town_Cherbourg,embark_town_Queenstown,embark_town_Southampton
0,0,3,22.0,1,0,7.2500,0,0,1,0,0,0,1
1,1,1,38.0,1,0,71.2833,1,1,0,0,1,0,0
2,1,3,26.0,0,0,7.9250,1,0,1,0,0,0,1
3,1,1,35.0,1,0,53.1000,1,1,0,0,0,0,1
4,0,3,35.0,0,0,8.0500,0,0,1,0,0,0,1


In [9]:
# We build X and y

y = data["survived"]
X = data.drop(columns=["survived"])
X.head()

,pclass,age,sibsp,parch,fare,sex_female,class_First,class_Third,who_child,embark_town_Cherbourg,embark_town_Queenstown,embark_town_Southampton
0,3,22.0,1,0,7.2500,0,0,1,0,0,0,1
1,1,38.0,1,0,71.2833,1,1,0,0,1,0,0
2,3,26.0,0,0,7.9250,1,0,1,0,0,0,1
3,1,35.0,1,0,53.1000,1,1,0,0,0,0,1
4,3,35.0,0,0,8.0500,0,0,1,0,0,0,1


In [10]:
# We MinMaxScale our features for you
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler().fit(X)
X_scaled = scaler.transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)
X.shape

(714, 12)

## 2. Düzenlileştirme olmadan Lojistik Regresyon

❓ Basit bir **düzenlileştirilmemiş** Lojistik Regresyon eğittikten sonra özellikleri önem sırasına göre azalan şekilde sıralayın (yani, eğitim sonrası katsayılara bakın)
- Dikkat: `LogisticRegression` varsayılan olarak cezalandırılmıştır
  - cezayı nasıl kaldıracağınızı öğrenmek için [penalty parametresine](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) bakın)
- Model yakınsayana kadar `max_iter`'i daha büyük bir sayıya çıkarın
- Çözücünün durma kriterini ayarlamak için `tol=1e-9` kullanın: gradyanın en büyük bileşeni bundan küçük olduğunda çözücü duracak. Daha yüksek değerlere ayarlarsanız, katsayıların `tol` değeriyle birlikte çok dalgalandığını görürsünüz.

<details>
    <summary>İpucu</summary>
    <img src="https://wagon-public-datasets.s3.amazonaws.com/data-science-images/05-ML/05-Model-Tuning/model_selection.png" alt="penalizing a regression" width="500">
</details>

In [13]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Veriyi oku
df = pd.read_csv('ML_titanic_dataset_encoded.csv')

# 2. Hedef ve özellikleri ayır (KÜÇÜK HARFLE 'survived' yazdık)
X = df.drop('survived', axis=1)
y = df['survived']

# 3. Eğitim ve test setini ayır
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Ölçeklendirme
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Veri başarıyla yüklendi! Sütun sayısı:", X.shape[1])

Veri başarıyla yüklendi! Sütun sayısı: 12


In [14]:
from sklearn.linear_model import LogisticRegression

# 1. Düzenlileştirilmemiş (Penalty yok) modeli kurma
# Not: scikit-learn'ün bazı sürümlerinde penalty=None yerine penalty='none' yazman gerekebilir.
log_reg_no_reg = LogisticRegression(
    penalty=None, 
    max_iter=10000, 
    tol=1e-9, 
    solver='lbfgs' # 'lbfgs' cezasız modellerde stabildir
)

# 2. Modeli eğitme
log_reg_no_reg.fit(X_train_scaled, y_train)

# 3. Özellikleri ve katsayılarını eşleştirme
# Katsayıların mutlak değerini alıyoruz çünkü negatif güçlü bir katsayı da çok önemlidir.
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': log_reg_no_reg.coef_[0],
    'Abs_Coefficient': abs(log_reg_no_reg.coef_[0])
})

# 4. Önem sırasına göre (azalan) sıralama
feature_importance_sorted = feature_importance.sort_values(by='Abs_Coefficient', ascending=False)

# Sonuçları göster
print(feature_importance_sorted)

                    Feature  Coefficient  Abs_Coefficient
11  embark_town_Southampton    -9.894160         9.894160
9     embark_town_Cherbourg    -9.067841         9.067841
10   embark_town_Queenstown    -4.719580         4.719580
5                sex_female     1.366606         1.366606
1                       age    -0.537608         0.537608
2                     sibsp    -0.426301         0.426301
6               class_First     0.420836         0.420836
8                 who_child     0.393094         0.393094
0                    pclass    -0.389197         0.389197
7               class_Third    -0.280996         0.280996
3                     parch    -0.127098         0.127098
4                      fare     0.079889         0.079889


❓`sex_female` katsayısının değerini sade Türkçe ile nasıl yorumlarsınız?

<details>
    <summary>Cevap</summary>

> "Diğer tüm şeyler eşitken (yaş, bilet sınıfı vb...),
kadın olmak hayatta kalma log-oranlarınızı 2.67 artırır (sizin katsayı değeriniz)"
    
> "Bu veri setinde mevcut olan diğer tüm açıklayıcı faktörleri kontrol ederken,
kadın olmak hayatta kalma oranlarınızı exp(2.67) = 14 kat artırır"

</details>

❓ Modelinize göre hayatta kalma şansını en çok etkileyen özellik hangisidir?  
Aşağıdaki `top_1_feature` listesini bu özelliğin adıyla doldurun

In [15]:
top_1_feature = ["embark_town_Southampton"]

In [16]:
from nbresult import ChallengeResult
result = ChallengeResult('unregularized', top_1_feature=top_1_feature)
result.write()
print(result.check())


============================= test session starts ==============================
platform linux -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /home/bariscan/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /home/bariscan/code/Workintech/S16D5-S-regularization/tests
plugins: typeguard-4.4.2, dash-4.0.0, anyio-4.8.0
collecting ... collected 1 item

test_unregularized.py::TestUnregularized::test_top_1 PASSED              [100%]

============================== 1 passed in 0.02s ===============================


💯 You can commit your code:

git add tests/unregularized.pickle

git commit -m 'Completed unregularized step'

git push origin master



## 3. L2 cezalı Lojistik Regresyon

Aşırı öğrenme olmadan **en önemli özellikleri** bulmak için log-kaybı **L2** terimi ile cezalandırılmış bir **Lojistik model** kullanalım.  
Bu, "Ridge" regresörünün "sınıflandırma" karşılığıdır

❓ **Güçlü düzenlileştirilmiş** bir `LogisticRegression` oluşturun ve özelliklerini önem sırasına göre sıralayın (katsayılara bakın)
- "Güçlü düzenlileştirilmiş" ile "Sklearn'in varsayılan düzenlileştirme faktöründen daha fazla" demek istiyoruz. 
- Sklearn'in varsayılan değerleri "ölçeklenmiş özellikler" için akılda tutulması gereken çok yararlı büyüklük mertebeleridir

In [17]:
# 1. Güçlü düzenlileştirilmiş L2 modeli (C değerini küçük seçiyoruz)
log_reg_l2 = LogisticRegression(
    penalty='l2', 
    C=0.01,        # Varsayılan 1.0'dan çok daha küçük, yani ceza çok daha güçlü
    max_iter=10000, 
    tol=1e-9, 
    solver='lbfgs',
    random_state=42
)

# 2. Modeli eğitme
log_reg_l2.fit(X_train_scaled, y_train)

# 3. Özellikleri önem sırasına göre sıralama
feature_importance_l2 = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': log_reg_l2.coef_[0],
    'Abs_Coefficient': abs(log_reg_l2.coef_[0])
})

feature_importance_l2_sorted = feature_importance_l2.sort_values(by='Abs_Coefficient', ascending=False)

# Sonuçları görelim
print(feature_importance_l2_sorted)

                    Feature  Coefficient  Abs_Coefficient
5                sex_female     0.622607         0.622607
0                    pclass    -0.200701         0.200701
1                       age    -0.194631         0.194631
7               class_Third    -0.184388         0.184388
6               class_First     0.172269         0.172269
8                 who_child     0.165723         0.165723
4                      fare     0.109386         0.109386
2                     sibsp    -0.097442         0.097442
9     embark_town_Cherbourg     0.072908         0.072908
11  embark_town_Southampton    -0.062159         0.062159
10   embark_town_Queenstown    -0.022917         0.022917
3                     parch     0.009891         0.009891


❓ Modelinize göre hayatta kalma şansını etkileyen ilk 2 özellik hangileridir?  
Aşağıdaki `top_2_features` listesini bu özelliklerin adlarıyla doldurun

In [18]:
top_2_features = ["sex_female", "pclass"]

#### 🧪 Kodunuzu aşağıda test edin

In [19]:
from nbresult import ChallengeResult
result = ChallengeResult('ridge', top_2=top_2_features)
result.write()
print(result.check())


============================= test session starts ==============================
platform linux -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /home/bariscan/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /home/bariscan/code/Workintech/S16D5-S-regularization/tests
plugins: typeguard-4.4.2, dash-4.0.0, anyio-4.8.0
collecting ... collected 1 item

test_ridge.py::TestRidge::test_top2 PASSED                               [100%]

============================== 1 passed in 0.02s ===============================


💯 You can commit your code:

git add tests/ridge.pickle

git commit -m 'Completed ridge step'

git push origin master



## 4. L1 cezalı Lojistik Regresyon

Bu sefer, **daha az önemli özellikleri filtrelemek** için log-kaybı **L1** terimi ile cezalandırılmış bir lojistik model kullanacağız.  
Bu, **Lasso** regresörünün "sınıflandırma" karşılığıdır

❓ **Güçlü düzenlileştirilmiş** bir `LogisticRegression` oluşturun ve özelliklerini önem sırasına göre sıralayın

In [20]:
from sklearn.linear_model import LogisticRegression

# 1. Güçlü düzenlileştirilmiş L1 (Lasso) modeli
# Lasso katsayıları tamamen sıfırlayarak özellik seçimi yapmamızı sağlar.
# C ne kadar küçükse, ceza o kadar güçlüdür ve daha fazla özellik elenir (sıfır olur).
log_reg_l1 = LogisticRegression(
    penalty='l1', 
    C=0.01,           # Güçlü düzenlileştirme için küçük C
    solver='liblinear', # L1 cezasını destekleyen solver
    max_iter=10000, 
    tol=1e-9,
    random_state=42
)

# 2. Modeli eğitme
log_reg_l1.fit(X_train_scaled, y_train)

# 3. Özellikleri önem sırasına göre sıralama
feature_importance_l1 = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': log_reg_l1.coef_[0],
    'Abs_Coefficient': abs(log_reg_l1.coef_[0])
})

feature_importance_l1_sorted = feature_importance_l1.sort_values(by='Abs_Coefficient', ascending=False)

# Sonuçları görelim
print(feature_importance_l1_sorted)

                    Feature  Coefficient  Abs_Coefficient
5                sex_female     0.417255         0.417255
0                    pclass     0.000000         0.000000
1                       age     0.000000         0.000000
2                     sibsp     0.000000         0.000000
3                     parch     0.000000         0.000000
4                      fare     0.000000         0.000000
6               class_First     0.000000         0.000000
7               class_Third     0.000000         0.000000
8                 who_child     0.000000         0.000000
9     embark_town_Cherbourg     0.000000         0.000000
10   embark_town_Queenstown     0.000000         0.000000
11  embark_town_Southampton     0.000000         0.000000


❓ L1 modelinize göre hayatta kalma şansı üzerinde kesinlikle hiçbir etkisi olmayan özellikler hangileridir?  
Aşağıdaki `zero_impact_features` listesini bu özelliklerin adlarıyla doldurun; listeye eleman eklemeniz gerekebilir.

- Bunlardan bazılarının düzenlileştirilmemiş modele göre "çok önemli" olduğunu fark ettiniz mi? 
- Bundan sonra doğrusal modellerimizi her zaman düzenlileştireceğiz!

In [21]:
zero_impact_features = [
    "pclass", "age", "sibsp", "parch", "fare", 
    "class_First", "class_Third", "who_child", 
    "embark_town_Cherbourg", "embark_town_Queenstown", "embark_town_Southampton"
]

#### 🧪 Kodunuzu aşağıda test edin

In [22]:
from nbresult import ChallengeResult
result = ChallengeResult('lasso', zero_impact_features = zero_impact_features)
result.write()
print(result.check())


============================= test session starts ==============================
platform linux -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /home/bariscan/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /home/bariscan/code/Workintech/S16D5-S-regularization/tests
plugins: typeguard-4.4.2, dash-4.0.0, anyio-4.8.0
collecting ... collected 1 item

test_lasso.py::TestLasso::test_zero_impact PASSED                        [100%]

============================== 1 passed in 0.01s ===============================


💯 You can commit your code:

git add tests/lasso.pickle

git commit -m 'Completed lasso step'

git push origin master



# 5. Bir adım geri çekilmek

🤯 **Bu katsayılardan bazıları neden başlangıçta bu kadar yüksekti?**

Düzenlileştirme ile kaldırılan üç özelliği düşünelim:
- `embark_town_Cherbourg`
- `embark_town_Southampton`
- `embark_town_Queenstown`

Üç biniş şehri tabii ki ilişkilidir: ikisinden binmediyseniz, üçüncüsünden binmiş olmalısınız. Yani biliyoruz ki: 

$$embark\_town\_Cherbourg + embark\_town\_Southampton + embark\_town\_Queenstown = 1$$

Bu üç özellik **mükemmel çoklu doğrusal bağıntılıdır**!

**Düzenlileştirilmemiş modeller kullanılırken, bu genellikle sayısal kararsızlığa yol açar**, ki burada gördüğümüz tam olarak buydu. Ayrıca böyle bir durumda elde ettiğimiz **katsayılara gerçekten güvenemeyeceğimiz** anlamına gelir.

❗️ Bu üç çoklu doğrusal bağıntılı özellik, `embark_town` kategorik özelliğinin one hot encoding'inden gelir.

Düzenlileştirme sayesinde bu sorunu aştık: üç şehir için katsayıların çok büyük olmasını engelledi. **İşte bu yüzden neredeyse her zaman düzenlileştirme kullanacağız.**

🔍 **Başlangıçta ayarladığımız `tol` parametresini hatırlıyor musunuz?**

Düzenlileştirmenin ekstra bir bonusu da `tol` ayarlamanın daha az önemli hale gelmesi: `1e-2` ve `1e-9` arasında herhangi bir değere ayarlayabilirsiniz ve katsayılar neredeyse hiç değişmez! 💪

**🏁 Tebrikler! Not defterinizi commit etmeyi ve push etmeyi unutmayın**